In [4]:
import sagemaker
import boto3

sess = sagemaker.Session()
bucket = sess.default_bucket()
region = boto3.Session().region_name
role = sagemaker.get_execution_role()

print(bucket)
print(region)
print(role)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
sagemaker-us-east-1-183914628679
us-east-1
arn:aws:iam::183914628679:role/LabRole


In [5]:
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model/model_credit_score.joblib", arcname="model_credit_score.joblib")
    tar.add("model/label_encoder.joblib", arcname="label_encoder.joblib")
    tar.add("model/feature_names.joblib", arcname="feature_names.joblib")

In [6]:
s3_uri = sess.upload_data(
    path="model.tar.gz",
    bucket=bucket,
    key_prefix="credit-score-model"
)

print(s3_uri)

s3://sagemaker-us-east-1-183914628679/credit-score-model/model.tar.gz


In [7]:
from sagemaker.sklearn.model import SKLearnModel

model = SKLearnModel(
    model_data=s3_uri,
    role=role,
    framework_version="1.4-2",
    py_version="py3",
    entry_point="inference.py"
)

In [8]:
endpoint_name = "credit-score-endpoint"

In [15]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name
)

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

-----!

In [16]:
import json

sample = {"instances": [[30, 50000, 4000, 5, 3, 10, 2, 5, 2, 5, 3, 1000, 30, 120, 1, 200, 500, 3000]]}

result = predictor.predict(sample)

print(result)

{'predictions': [2], 'labels': ['Standard']}


In [14]:
import boto3

sm = boto3.client("sagemaker")

response = sm.list_endpoints()
print(response["Endpoints"])

[]


In [13]:
import boto3

sm_client = boto3.client("sagemaker", region_name="us-east-1")
ENDPOINT_NAME = "credit-score-endpoint"

# 1. Delete the stuck endpoint
print(f"Deleting failed endpoint: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint deletion triggered.")
except Exception as e:
    print(f"No endpoint found to delete: {e}")

# 2. Delete the conflicting endpoint configuration
print(f"Deleting endpoint configuration: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print("Endpoint configuration deletion triggered.")
except Exception as e:
    print(f"No config found to delete: {e}")

print("\nCleanup complete! You can now safely run your main deploy script.")

Deleting failed endpoint: credit-score-endpoint...
Endpoint deletion triggered.
Deleting endpoint configuration: credit-score-endpoint...
Endpoint configuration deletion triggered.

Cleanup complete! You can now safely run your main deploy script.
